In [1]:
"""
MiCRM v10 dataset generator.

Unchanged in its ecology, ODE, cross-feeding, suppression
formula, and OUTPUT COLUMN FORMAT. Only three things differ, all documented below:

  FIX 1 : many distinct broths instead of 20.
      N_BROTHS_PER_LEVEL controls how many random nutrient environments exist per
      diversity level. With 20 per level you get 100 distinct broths (vs 20 in v9),
      so R0 is no longer a 20-category label and GroupKFold over broths can generalise.

  FIX 2 (most crucial): CONSTANT_ENV.
      If True, the resource-quality w, leakage l, and dilution delta are drawn ONCE and
      shared by every broth, so R0 is the only thing that changes between environments.
      Set False to recover the exact v9 behaviour (per-broth random w, l, delta).

  SPEEDUP (same math): the cross-feeding term is computed with np.einsum instead of a
      Python triple loop. Results are identical up to solver tolerance, just much faster.

The three output CSVs (community_dataset, community_index, monoculture_dataset) have the
SAME columns as v9, so our existing XGBoost / LightGBM / CatBoost code reads them as is.
"""

import numpy as np
import pandas as pd
import warnings, os, csv
warnings.filterwarnings("ignore")
from scipy.integrate import solve_ivp
from tqdm import tqdm

rng = np.random.default_rng(seed=42)

N_SPECIES   = 101
N_RESOURCES = 20
N_GENES     = 8
PATHOGEN_IDX = 0
N_COMMUNITY  = N_SPECIES - 1
N_SPECIALIST = N_COMMUNITY // 2
N_GENERALIST = N_COMMUNITY - N_SPECIALIST

T_MEASURE = 14.0
DIVERSITY_LEVELS = [1, 2, 4, 8, 16]

# FIX 1: many broths per level 
N_BROTHS_PER_LEVEL = 30                      # 30 x 5 levels = 150 distinct broths
N_REPLICATES_PER_LEVEL = {lv: N_BROTHS_PER_LEVEL for lv in DIVERSITY_LEVELS}
N_BROTHS = sum(N_REPLICATES_PER_LEVEL.values())

# FIX 2: hold the environment constant across broths
CONSTANT_ENV = True

TOTAL_CARBON_SUPPLY = 20.0
COMMUNITY_SIZES = [12, 16, 24, 32, 48]
N_SAMPLES_PER_SIZE = 100                       # per (broth x size); tune for scale

UPTAKE_SPARSITY = {"pathogen": (0.45, 0.70), "specialist": (0.10, 0.25), "generalist": (0.40, 0.65)}
CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX = 2, 6

RESOURCE_NAMES = [f"c{k+1}" for k in range(N_RESOURCES)]
SPECIES_NAMES  = [f"Species_{i+1:04d}" for i in range(N_SPECIES)]

OUTPUT_DIR = r"C:\Users\Pratham Shah\Downloads\Tubingen\Codes\MiCRM_Final_Dataset"


def build_species_traits():
    """Same as v9. Genome -> sigmoid(genes @ W + bias) -> mask to active nutrients = c_matrix."""
    genes = rng.integers(0, 2, size=(N_SPECIES, N_GENES))
    W = rng.normal(0.0, 0.5, size=(N_GENES, N_RESOURCES))
    c_matrix = np.zeros((N_SPECIES, N_RESOURCES))
    uptake_mask = np.zeros((N_SPECIES, N_RESOURCES), dtype=bool)
    species_type, preferred_nutrient = [], []
    for i in range(N_SPECIES):
        if i == PATHOGEN_IDX:
            stype, pref = "pathogen", -1
        elif 1 <= i <= N_SPECIALIST:
            stype, pref = "specialist", (i - 1) % N_RESOURCES
        else:
            stype, pref = "generalist", -1
        species_type.append(stype); preferred_nutrient.append(pref)
        lo, hi = UPTAKE_SPARSITY[stype]
        n_active = max(1, int(round(rng.uniform(lo, hi) * N_RESOURCES)))
        active = rng.choice(N_RESOURCES, size=n_active, replace=False).tolist()
        if pref >= 0 and pref not in active:
            active[0] = pref
        uptake_mask[i, active] = True
        if stype == "specialist":
            bias = rng.uniform(-2.0, -1.0, size=N_RESOURCES); bias[pref] += rng.uniform(2.5, 4.0)
        else:
            bias = rng.uniform(-0.8, -0.3, size=N_RESOURCES)
        raw = genes[i].astype(float) @ W + bias
        c_matrix[i] = (1.0 / (1.0 + np.exp(-raw))) * uptake_mask[i]
    return genes, c_matrix, uptake_mask, species_type, preferred_nutrient


def build_broths():
    """v9 diversity-level design, but many broths per level. Optionally constant w, l, delta."""
    shared_w = rng.uniform(0.6, 1.2, size=N_RESOURCES)
    shared_l = rng.uniform(0.1, 0.3, size=N_RESOURCES)
    shared_delta = float(rng.uniform(0.05, 0.20))
    broths, broth_idx = [], 0
    for n_active in DIVERSITY_LEVELS:
        for rep in range(N_REPLICATES_PER_LEVEL[n_active]):
            active_set = sorted(rng.choice(N_RESOURCES, size=n_active, replace=False).tolist())
            R0 = np.zeros(N_RESOURCES)
            for k in active_set:
                R0[k] = (TOTAL_CARBON_SUPPLY / n_active) * rng.uniform(0.85, 1.15)
            if R0.sum() > 1e-12:
                R0 = R0 / R0.sum() * TOTAL_CARBON_SUPPLY
            frac_R0 = R0 / (R0.sum() + 1e-12)
            active_flag = np.array([1 if k in active_set else 0 for k in range(N_RESOURCES)], dtype=int)
            if CONSTANT_ENV:
                w, l, delta = shared_w, shared_l, shared_delta
            else:
                w = rng.uniform(0.6, 1.2, size=N_RESOURCES)
                l = rng.uniform(0.1, 0.3, size=N_RESOURCES)
                delta = float(rng.uniform(0.05, 0.20))
            broths.append({"name": f"Broth_{broth_idx+1:03d}", "diversity_level": n_active,
                           "replicate": rep + 1, "active_set": active_set, "R0": R0,
                           "frac_R0": frac_R0, "active_flag": active_flag,
                           "w": w, "l": l, "delta": delta})
            broth_idx += 1
    return broths


def build_species_crossfeed_matrices(species_type):
    """Same as v9, but stacked into one (S, M, M) array so the ODE can use einsum."""
    D = np.zeros((N_SPECIES, N_RESOURCES, N_RESOURCES))
    for i in range(N_SPECIES):
        for beta in range(N_RESOURCES):
            n_out = int(rng.integers(CROSSFEED_OUT_MIN, CROSSFEED_OUT_MAX + 1))
            active_alpha = rng.choice(N_RESOURCES, size=n_out, replace=False)
            if species_type[i] == "specialist":
                pref = (i - 1) % N_RESOURCES
                alpha_vec = np.array([rng.uniform(2.0, 5.0) if a == pref and beta != pref
                                      else rng.uniform(0.5, 2.0) for a in active_alpha])
            elif species_type[i] == "pathogen":
                alpha_vec = rng.uniform(0.5, 3.0, size=n_out)
            else:
                alpha_vec = rng.uniform(0.8, 1.5, size=n_out)
            D[i, active_alpha, beta] = rng.dirichlet(alpha_vec)
    return D


def micrm_ode(t, y, c, w, l, m, g_rate, delta, R0, D_sub):
    """Same equations as v9. Cross-feeding vectorised with einsum (identical math)."""
    n_sp = c.shape[0]
    N = np.maximum(y[:n_sp], 0.0)
    R = np.maximum(y[n_sp:], 0.0)
    energy_intake = c @ (w * (1.0 - l) * R)
    dN = N * (g_rate * (energy_intake - m)) - delta * N
    uptake = (c.T @ N) * R
    leaked = (l * c * R[None, :]) * N[:, None]        # (n_sp, M): flux leaked by i from beta
    crossfeed = np.einsum("iab,ib->a", D_sub, leaked) # redistribute to alpha
    dR = delta * (R0 - R) - uptake + crossfeed
    return np.concatenate([dN, dR])


def run_micrm(c_full, w, l, m_full, g_full, delta, R0, D, species_subset, t_end=T_MEASURE):
    """Same as v9. Integrate one community to T_MEASURE with RK45."""
    idx = np.array(species_subset)
    c_sub, m_sub, g_sub, D_sub = c_full[idx], m_full[idx], g_full[idx], D[idx]
    y0 = np.concatenate([np.full(idx.size, 0.01), R0.copy()])
    sol = solve_ivp(micrm_ode, (0.0, t_end), y0,
                    args=(c_sub, w, l, m_sub, g_sub, delta, R0, D_sub),
                    method="RK45", t_eval=np.linspace(0, t_end, max(50, int(t_end * 5))),
                    rtol=1e-5, atol=1e-7)
    y_final = sol.y[:, -1]
    N_out = np.zeros(N_SPECIES)
    for local_idx, global_idx in enumerate(species_subset):
        N_out[global_idx] = max(0.0, y_final[local_idx])
    return N_out, np.maximum(y_final[idx.size:], 0.0), sol


def calc_suppression(N_path_measured, N_path_solo):
    eps = 1e-12
    if N_path_solo < eps:
        return 0.0
    return float(np.clip(1.0 - N_path_measured / (N_path_solo + eps), 0.0, 1.0))


def _lum_transform(supp_score, N_path_solo):
    ratio = max(1.0 - supp_score, 1e-10)
    return round(float(np.clip(np.log10(ratio), -10.0, 0.0)), 6)


def generate_dataset(output_dir):
    """
    instead of collecting every row in a Python list (which will cause memory error), 
    each row is written straight to disk as soon as it is
    produced, and the small community-level index is built directly during
    the loop instead of being pivoted out of the giant file afterwards.
    """
    
    print("  MiCRM v10 -- many-broths fix, v9 format, memory-safe writer")
    print(f"  Distinct broths: {N_BROTHS}   CONSTANT_ENV: {CONSTANT_ENV}")
    total_comm = N_BROTHS * len(COMMUNITY_SIZES) * N_SAMPLES_PER_SIZE
    mean_size = sum(COMMUNITY_SIZES) / len(COMMUNITY_SIZES)
    print(f"  Expected communities: {total_comm:,}")
    print(f"  Expected community_dataset rows (long format): ~{int(total_comm*mean_size):,}")


    genes, c_matrix, uptake_mask, species_type, preferred_nutrient = build_species_traits()
    broths = build_broths()
    D = build_species_crossfeed_matrices(species_type)

    m = rng.uniform(0.1, 0.5, size=N_SPECIES)
    g = rng.uniform(0.5, 1.5, size=N_SPECIES)
    m[PATHOGEN_IDX] = float(rng.uniform(0.05, 0.12))

    pathogen_solo = {}
    for b in tqdm(broths, desc="  Pathogen solo"):
        N_s, _, _ = run_micrm(c_matrix, b["w"], b["l"], m, g, b["delta"], b["R0"], D, [PATHOGEN_IDX])
        pathogen_solo[b["name"]] = float(N_s[PATHOGEN_IDX])

    ss_lum = {}
    strain_indices = list(range(1, N_SPECIES))
    for b in tqdm(broths, desc="  Pairwise ODE"):
        ss_lum[b["name"]] = {}
        for sp in strain_indices:
            N_pair, _, _ = run_micrm(c_matrix, b["w"], b["l"], m, g, b["delta"], b["R0"], D, [sp, PATHOGEN_IDX])
            ss_lum[b["name"]][sp] = calc_suppression(N_pair[PATHOGEN_IDX], pathogen_solo[b["name"]])

    os.makedirs(output_dir, exist_ok=True)
    community_csv_path = os.path.join(output_dir, "community_dataset_v10.csv")


    community_meta_rows = []
    cid = 0
    writer = None  

    with open(community_csv_path, "w", newline="") as f:
        for b in tqdm(broths, desc="  Community ODE"):
            bname, N_solo_p, dlevel = b["name"], pathogen_solo[b["name"]], b["diversity_level"]
            for comm_size in COMMUNITY_SIZES:
                for _ in range(N_SAMPLES_PER_SIZE):
                    members = sorted(rng.choice(strain_indices, size=comm_size, replace=False).tolist())
                    N_comm, _, _ = run_micrm(c_matrix, b["w"], b["l"], m, g, b["delta"], b["R0"], D,
                                             members + [PATHOGEN_IDX])
                    comm_supp = calc_suppression(N_comm[PATHOGEN_IDX], N_solo_p)
                    comm_id = f"Comm_{cid:06d}"; cid += 1

                    # (A) the small community-level row
                    meta_row = {"community_id": comm_id, "broth_id": bname, "diversity_level": dlevel,
                               "community_size": comm_size, "community_suppression": round(comm_supp, 6),
                               "delta_per_h": round(b["delta"], 4), "R0_sum": round(float(b["R0"].sum()), 4),
                               "n_members": comm_size}
                    for k in range(N_RESOURCES):
                        meta_row[f"R0_c{k+1}"] = round(float(b["R0"][k]), 4)
                        meta_row[f"frac_c{k+1}"] = round(float(b["frac_R0"][k]), 6)
                        meta_row[f"active_c{k+1}"] = int(b["active_flag"][k])
                    for sp_name in SPECIES_NAMES[1:]:
                        meta_row[sp_name] = 0
                    for sp in members:
                        meta_row[SPECIES_NAMES[sp]] = 1
                    community_meta_rows.append(meta_row)

                    # (B) the large per-member rows
                    for sp in members:
                        row = {"community_id": comm_id, "broth_id": bname, "diversity_level": dlevel,
                               "community_size": comm_size, "community_suppression": round(comm_supp, 6),
                               "delta_per_h": round(b["delta"], 4), "R0_sum": round(float(b["R0"].sum()), 4),
                               "strain_id": SPECIES_NAMES[sp], "species_type": species_type[sp],
                               "maintenance_cost": round(float(m[sp]), 4), "growth_rate": round(float(g[sp]), 4),
                               "ss_luminescence": _lum_transform(ss_lum[bname][sp], N_solo_p),
                               "ss_suppression_raw": round(ss_lum[bname][sp], 6)}
                        for gi in range(N_GENES):
                            row[f"gene_{gi+1}"] = int(genes[sp, gi])
                        for k in range(N_RESOURCES):
                            row[f"uptake_c{k+1}"] = round(float(c_matrix[sp, k]), 6)
                        for k in range(N_RESOURCES):
                            row[f"R0_c{k+1}"] = round(float(b["R0"][k]), 4)
                            row[f"frac_c{k+1}"] = round(float(b["frac_R0"][k]), 6)
                            row[f"active_c{k+1}"] = int(b["active_flag"][k])

                        if writer is None:
                            writer = csv.DictWriter(f, fieldnames=list(row.keys()))
                            writer.writeheader()
                        writer.writerow(row)

    df_idx = pd.DataFrame(community_meta_rows)
    return df_idx, community_csv_path, genes, c_matrix, m, g, broths, species_type, ss_lum, pathogen_solo, D


def build_monoculture_dataset(genes, c_matrix, m, g, broths, species_type, ss_lum, pathogen_solo):
    """Same as v9. One row per strain."""
    ref = broths[0]
    rows = []
    for sp in range(1, N_SPECIES):
        all_supp = [ss_lum[b["name"]][sp] for b in broths]
        row = {"strain_id": SPECIES_NAMES[sp], "species_type": species_type[sp],
               "maintenance_cost": round(float(m[sp]), 4), "growth_rate": round(float(g[sp]), 4),
               "ss_luminescence": _lum_transform(float(np.mean(all_supp)),
                                                 np.mean([pathogen_solo[b["name"]] for b in broths])),
               "ss_suppression_mean": round(float(np.mean(all_supp)), 6)}
        for lv in DIVERSITY_LEVELS:
            supp = [ss_lum[b["name"]][sp] for b in broths if b["diversity_level"] == lv]
            row[f"ss_supp_d{lv}"] = round(float(np.mean(supp)), 6)
        for gi in range(N_GENES):
            row[f"gene_{gi+1}"] = int(genes[sp, gi])
        for k in range(N_RESOURCES):
            row[f"uptake_c{k+1}"] = round(float(c_matrix[sp, k]), 6)
        for k in range(N_RESOURCES):
            row[f"R0_c{k+1}"] = round(float(ref["R0"][k]), 4)
        rows.append(row)
    return pd.DataFrame(rows)


# plots
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TEAL, ORANGE, NAVY, GOLD, RED, GREY = "#1a6b6b", "#d4720c", "#1c2e4a", "#c8991f", "#9c2a2a", "#8a97a8"
LEVEL_COLORS = {1: "#2196F3", 2: "#4CAF50", 4: "#FF9800", 8: "#E91E63", 16: "#9C27B0"}


def _trajectory(c_matrix, m, g, D, broth, members, t_end=120.0):
    """Run one community in one broth and return the FULL time trajectory (sol)."""
    subset = list(members) + [PATHOGEN_IDX]
    _, _, sol = run_micrm(c_matrix, broth["w"], broth["l"], m, g,
                          broth["delta"], broth["R0"], D, subset, t_end=t_end)
    return subset, sol


def make_dynamics_figures(c_matrix, m, g, D, broths, species_type, pathogen_solo, OUTPUT_DIR):
    """New figures that visualise the RAW differential-equation dynamics over time."""
    broth = max(broths, key=lambda b: b["diversity_level"] if b["diversity_level"] <= 8 else -1)
    rng_local = np.random.default_rng(0)
    members = sorted(rng_local.choice(range(1, N_SPECIES), size=16, replace=False).tolist())
    t_end = 120.0

    subset, sol = _trajectory(c_matrix, m, g, D, broth, members, t_end)
    _, sol_solo = _trajectory(c_matrix, m, g, D, broth, [], t_end)
    n_sp = len(subset)
    tt = sol.t
    path_row = subset.index(PATHOGEN_IDX)

    # Fig D1: species abundance trajectories
    fig, ax = plt.subplots(figsize=(9, 5))
    for k, gi in enumerate(subset):
        if gi == PATHOGEN_IDX:
            continue
        color = TEAL if species_type[gi] == "specialist" else "#4CAF50"
        ax.plot(tt, sol.y[k], color=color, lw=1.0, alpha=0.6)
    ax.plot(tt, sol.y[path_row], color=RED, lw=3.0, label="pathogen", zorder=5)
    ax.plot([], [], color=TEAL, lw=1.5, label="specialists")
    ax.plot([], [], color="#4CAF50", lw=1.5, label="generalists")
    ax.axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8, label=f"T_MEASURE={T_MEASURE}")
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_title(f"Dynamics D1 - Species abundances over time  ({broth['name']}, D={broth['diversity_level']})",
                 fontweight="bold", color=NAVY)
    ax.set_xlabel("time"); ax.set_ylabel("abundance N_i(t)  (symlog)")
    ax.legend(fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "figD1_species_dynamics.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig D2: resource dynamics (drawdown + cross-feeding) 
    fig, ax = plt.subplots(figsize=(9, 5))
    R_traj = sol.y[n_sp:]
    active = set(broth["active_set"])
    for a in range(N_RESOURCES):
        if a in active:
            ax.plot(tt, R_traj[a], color=TEAL, lw=1.4, alpha=0.7)
        else:
            ax.plot(tt, R_traj[a], color=ORANGE, lw=1.2, ls="--", alpha=0.7)
    ax.plot([], [], color=TEAL, lw=1.6, label="supplied nutrients (R0>0)")
    ax.plot([], [], color=ORANGE, lw=1.4, ls="--", label="absent nutrients (appear via cross-feeding)")
    ax.axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8, label=f"T_MEASURE={T_MEASURE}")
    ax.set_title("Dynamics D2 - Resource concentrations over time", fontweight="bold", color=NAVY)
    ax.set_xlabel("time"); ax.set_ylabel("resource R_alpha(t)")
    ax.legend(fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "figD2_resource_dynamics.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig D3: pathogen alone vs in community
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(sol_solo.t, sol_solo.y[0], color=RED, lw=2.6, label="pathogen ALONE")
    ax.plot(tt, sol.y[path_row], color=NAVY, lw=2.6, label="pathogen IN COMMUNITY")
    ax.axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8, label=f"T_MEASURE={T_MEASURE}")
    ax.set_title("Dynamics D3 - How suppression emerges", fontweight="bold", color=NAVY)
    ax.set_xlabel("time"); ax.set_ylabel("pathogen abundance")
    ax.legend(fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "figD3_alone_vs_community.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig D4: T_measure sensitivity
    solo_interp = np.interp(tt, sol_solo.t, sol_solo.y[0])
    supp_t = np.clip(1.0 - sol.y[path_row] / (solo_interp + 1e-12), 0, 1)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].plot(tt, supp_t, color=TEAL, lw=2.2)
    ax[0].axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8, label=f"T_MEASURE={T_MEASURE}")
    ax[0].set_title("Suppression vs measurement time", fontweight="bold", color=NAVY)
    ax[0].set_xlabel("measurement time T"); ax[0].set_ylabel("suppression at T"); ax[0].legend(fontsize=9, frameon=False)
    ax[1].plot(tt, sol.y[path_row], color=NAVY, lw=2.2, label="in community")
    ax[1].plot(tt, solo_interp, color=RED, lw=2.2, label="solo")
    ax[1].axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8)
    ax[1].set_title("Pathogen abundance vs time", fontweight="bold", color=NAVY)
    ax[1].set_xlabel("measurement time T"); ax[1].set_ylabel("pathogen abundance"); ax[1].legend(fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "figD4_Tmeasure_sensitivity.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig D5: pathogen net per-capita growth rate over time (R-star link) 
    R_traj = sol.y[n_sp:]
    c_path = c_matrix[PATHOGEN_IDX]
    intake_t = (broth["w"] * (1 - broth["l"]) * R_traj.T) @ c_path
    net_growth = g[PATHOGEN_IDX] * (intake_t - m[PATHOGEN_IDX]) - broth["delta"]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.axhline(0, color="black", lw=0.9)
    ax.plot(tt, net_growth, color=RED, lw=2.4)
    ax.fill_between(tt, net_growth, 0, where=(net_growth < 0), color=RED, alpha=0.12)
    ax.axvline(T_MEASURE, color=GOLD, ls="--", lw=1.8, label=f"T_MEASURE={T_MEASURE}")
    ax.set_title("Dynamics D5 - Pathogen net growth rate (below 0 => being excluded)",
                 fontweight="bold", color=NAVY)
    ax.set_xlabel("time"); ax.set_ylabel("pathogen per-capita net growth rate")
    ax.legend(fontsize=9, frameon=False)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "figD5_pathogen_growth_rate.png"), dpi=150, bbox_inches="tight"); plt.close(fig)


def make_diagnostic_figures(community_csv_path, df_idx, c_matrix, species_type, broths, pathogen_solo, OUTPUT_DIR):

    #  Fig 1: dataset quality 
    ind_supp = pd.read_csv(community_csv_path, usecols=["ss_suppression_raw"])["ss_suppression_raw"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].hist(df_idx["community_suppression"], bins=40, color=TEAL, edgecolor="white")
    ax[0].set_title("Community suppression distribution"); ax[0].set_xlabel("suppression"); ax[0].set_ylabel("count")
    ax[1].hist(ind_supp, bins=40, color=ORANGE, edgecolor="white")
    ax[1].set_title("Individual (pairwise) suppression"); ax[1].set_xlabel("ss_suppression_raw")
    levels = DIVERSITY_LEVELS
    mean_solo = [np.mean([pathogen_solo[b["name"]] for b in broths if b["diversity_level"] == lv]) for lv in levels]
    ax[2].bar([str(l) for l in levels], mean_solo, color=TEAL, edgecolor="white")
    ax[2].set_title("Pathogen solo abundance by diversity level"); ax[2].set_xlabel("diversity level")
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig1_dataset_quality.png"), dpi=150, bbox_inches="tight"); plt.close(fig)
    del ind_supp

    # Fig 2: suppression vs size and diversity 
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    grp = df_idx.groupby("community_size")["community_suppression"]
    ax[0].errorbar(grp.mean().index, grp.mean().values, yerr=grp.std().values,
                   color=TEAL, lw=2, marker="o", capsize=4)
    ax[0].set_title("Suppression vs community size"); ax[0].set_xlabel("community size N"); ax[0].set_ylabel("mean suppression")
    piv = df_idx.pivot_table(index="community_size", columns="diversity_level",
                             values="community_suppression", aggfunc="mean")
    im = ax[1].imshow(piv.values, cmap="YlOrRd", aspect="auto", origin="lower")
    ax[1].set_xticks(range(len(piv.columns))); ax[1].set_xticklabels(piv.columns)
    ax[1].set_yticks(range(len(piv.index))); ax[1].set_yticklabels(piv.index)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax[1].text(j, i, f"{piv.values[i,j]:.2f}", ha="center", va="center", fontsize=8)
    ax[1].set_title("Mean suppression: size x diversity"); ax[1].set_xlabel("diversity level"); ax[1].set_ylabel("community size N")
    fig.colorbar(im, ax=ax[1])
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig2_suppression_size_diversity.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig 3: suppression vs diversity level (boxplot)
    fig, ax = plt.subplots(figsize=(8, 5))
    data = [df_idx[df_idx["diversity_level"] == lv]["community_suppression"].values for lv in DIVERSITY_LEVELS]
    ax.boxplot(data, labels=[str(l) for l in DIVERSITY_LEVELS])
    ax.set_title("Suppression vs resource diversity level", fontweight="bold", color=NAVY)
    ax.set_xlabel("diversity level (# active nutrients)"); ax.set_ylabel("community suppression")
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig3_suppression_by_diversity.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig 4: uptake matrix c_i,alpha
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(c_matrix, cmap="YlOrRd", aspect="auto")
    ax.set_title("Uptake matrix c[i, alpha]  (white = cannot eat)", fontweight="bold", color=NAVY)
    ax.set_xlabel("nutrient alpha"); ax.set_ylabel("species i"); fig.colorbar(im, ax=ax)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig4_uptake_matrix.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig 5: R0 matrix across broths
    R0_mat = np.array([b["R0"] for b in broths])
    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(R0_mat, cmap="viridis", aspect="auto")
    ax.set_title(f"R0 matrix: {len(broths)} broths x {N_RESOURCES} nutrients", fontweight="bold", color=NAVY)
    ax.set_xlabel("nutrient"); ax.set_ylabel("broth (grouped by diversity level)"); fig.colorbar(im, ax=ax)
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig5_R0_matrix.png"), dpi=150, bbox_inches="tight"); plt.close(fig)

    # Fig 6: individual vs community suppression
    pair_cols = pd.read_csv(community_csv_path, usecols=["community_id", "ss_suppression_raw"])
    per_comm = pair_cols.groupby("community_id")["ss_suppression_raw"].mean().reset_index(name="mean_ind")
    per_comm = per_comm.merge(df_idx[["community_id", "community_suppression"]], on="community_id")
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(per_comm["mean_ind"], per_comm["community_suppression"], s=8, alpha=0.3, color=TEAL)
    ax.set_title("Individual vs community suppression", fontweight="bold", color=NAVY)
    ax.set_xlabel("mean individual suppression of members"); ax.set_ylabel("community suppression")
    plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig6_individual_vs_community.png"), dpi=150, bbox_inches="tight"); plt.close(fig)
    del pair_cols, per_comm


def make_figures(community_csv_path, df_idx, df_mono, broths, pathogen_solo, c_matrix, m, g, D, species_type, OUTPUT_DIR):
    """Master call: v9-style diagnostics + new raw-dynamics figures."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    make_diagnostic_figures(community_csv_path, df_idx, c_matrix, species_type, broths, pathogen_solo, OUTPUT_DIR)
    make_dynamics_figures(c_matrix, m, g, D, broths, species_type, pathogen_solo, OUTPUT_DIR)
    print("  figures written to", OUTPUT_DIR)


if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    (df_idx, community_csv_path, genes, c_matrix, m, g, broths,
     species_type, ss_lum, pathogen_solo, D) = generate_dataset(OUTPUT_DIR)
    df_mono = build_monoculture_dataset(genes, c_matrix, m, g, broths, species_type, ss_lum, pathogen_solo)
    df_idx.to_csv(os.path.join(OUTPUT_DIR, "community_index_v10.csv"), index=False)
    df_mono.to_csv(os.path.join(OUTPUT_DIR, "monoculture_dataset_v10.csv"), index=False)
    print(f"community_dataset_v10.csv (streamed) : rows -> see file on disk")
    print(f"community_index_v10.csv   : {df_idx.shape}")
    print(f"monoculture_dataset_v10.csv: {df_mono.shape}")
    print("distinct broths:", df_idx["broth_id"].nunique())
    make_figures(community_csv_path, df_idx, df_mono, broths, pathogen_solo,
                 c_matrix, m, g, D, species_type, OUTPUT_DIR)

  MiCRM v10 -- many-broths fix, v9 format, memory-safe writer
  Distinct broths: 150   CONSTANT_ENV: True
  Expected communities: 75,000
  Expected community_dataset rows (long format): ~1,980,000


  Community ODE: 100%|█████████████████████████████████████████████████████████████| 150/150 [1:00:08<00:00, 24.06s/it]


community_dataset_v10.csv (streamed) : rows -> see file on disk
community_index_v10.csv   : (75000, 168)
monoculture_dataset_v10.csv: (100, 59)
distinct broths: 150
  figures written to C:\Users\Pratham Shah\Downloads\Tubingen\Codes\MiCRM_Final_Dataset
